# T2.L7 — Використання систем комп’ютерної математики в наукових дослідженнях

**Mini-research project:** question → hypothesis → data → calibration → verification → sensitivity → uncertainty → conclusion.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

LESSON_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = LESSON_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from model import analytical_trajectory, numerical_trajectory, calibrate_parameters, scenario_batch, bootstrap_calibration, quantile_summary, time_to_threshold

## 1. Research question and hypothesis

**Question:** наскільки надійно за шумними спостереженнями можна оцінити параметри `q` та `k` і спрогнозувати час досягнення порогу `S=80`?

**Hypothesis:** calibrated model відновить параметри поблизу істинних значень, але прогноз матиме ненульову uncertainty.

In [ ]:
config = json.loads((LESSON_DIR/'experiment_config.json').read_text(encoding='utf-8'))
data = pd.read_csv(LESSON_DIR/'data'/'synthetic_observations.csv')
data

## 2. Mathematical model

\[\frac{dS}{dt}=q-kS,\quad S(0)=S_0\]

\[S(t)=\frac{q}{k}+(S_0-\frac{q}{k})e^{-kt}\]

In [ ]:
t = data['time'].to_numpy(float)
y = data['observed_s'].to_numpy(float)
fit = calibrate_parameters(t, y, s0=config['s0'])
fit

In [ ]:
fitted = analytical_trajectory(t, s0=config['s0'], q=fit['q'], k=fit['k'])
plt.scatter(t, y, label='observations')
plt.plot(t, fitted, label='fitted model')
plt.xlabel('time'); plt.ylabel('S(t)'); plt.legend(); plt.show()

## 3. Independent numerical verification

In [ ]:
numerical = numerical_trajectory(t, s0=config['s0'], q=fit['q'], k=fit['k'])
verification_error = np.max(np.abs(fitted - numerical))
verification_error

## 4. Point prediction

In [ ]:
tt = time_to_threshold(s0=config['s0'], q=fit['q'], k=fit['k'], threshold=config['threshold'])
s20 = analytical_trajectory([config['horizon']], s0=config['s0'], q=fit['q'], k=fit['k'])[0]
{'time_to_threshold': tt, 'S(horizon)': s20}

## 5. Scenario / sensitivity experiment

In [ ]:
q_values = [fit['q']*m for m in config['q_multipliers']]
scenarios = scenario_batch(q_values, s0=config['s0'], k=fit['k'], horizon=config['horizon'], threshold=config['threshold'])
scenarios.insert(0,'q_multiplier',config['q_multipliers'])
scenarios

In [ ]:
plt.plot(scenarios['q_multiplier'], scenarios['time_to_threshold'], marker='o')
plt.xlabel('q multiplier'); plt.ylabel('time to threshold'); plt.show()

## 6. Bootstrap uncertainty

In [ ]:
boot = bootstrap_calibration(t, y, s0=config['s0'], threshold=config['threshold'], horizon=config['horizon'], n_boot=300, seed=config['seed'])
quantile_summary(boot, 'time_to_threshold')

In [ ]:
plt.hist(boot['time_to_threshold'], bins=25)
plt.xlabel('time to threshold'); plt.ylabel('frequency'); plt.show()

## 7. Interpretation

1. Point estimate не є єдиним можливим результатом.
2. Sensitivity показує залежність висновку від `q`.
3. Bootstrap характеризує uncertainty за прийнятої моделі та resampling scheme.
4. Verification підтверджує реалізацію, але не validation.

## 8. Research transfer

Перенесіть цей workflow на власну дисертаційну модель.